# AlphaTrain Pillar 2g: Hybrid Interleaved Training (Colab)

**Interleaved two-dataloader training:**
- Expert batches: `pol_CE + 0.001 * (ranking_loss + anchor_MSE)` — keeps value head sharp
- Self-play batches: `pol_CE + 0.001 * MSE(value, TD_return)` — teaches MCTS positions
- Losses averaged: `(loss_expert + loss_selfplay) / 2`

**Parameters:** T=0.3 sharpening, LR=5e-5, 15 epochs, val_weight=0.001

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `selfplay_iter2.pt.gz` — self-play data (sharpened T=0.3)
3. `alphatrain_pairwise.pt` — expert pairwise data (already on Drive)
4. `pillar2f_best.pt` — base model (already on Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy model + expert data
os.makedirs('/content/alphatrain/data', exist_ok=True)
for f in ['pillar2f_best.pt', 'alphatrain_pairwise.pt']:
    src = os.path.join(DRIVE, f)
    dst = os.path.join('/content/alphatrain/data', f)
    if not os.path.exists(dst):
        print(f'Copying {f}...')
        shutil.copy(src, dst)
    print(f'{f}: {os.path.getsize(dst)/1e6:.0f} MB')

# Decompress self-play data
gz_src = os.path.join(DRIVE, 'selfplay_iter2.pt.gz')
pt_dst = '/content/alphatrain/data/selfplay_iter2.pt'
if not os.path.exists(pt_dst):
    print('Decompressing selfplay_iter2.pt.gz...')
    t0 = time.time()
    !gzip -dc {gz_src} > {pt_dst}
    print(f'Done in {time.time()-t0:.0f}s: {os.path.getsize(pt_dst)/1e9:.1f} GB')
else:
    print(f'selfplay_iter2.pt: {os.path.getsize(pt_dst)/1e9:.1f} GB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')
    if g.total_memory < 30e9:
        print('WARNING: Need A100/H100 for hybrid training (two dataloaders).')

# Quick sanity checks
!cd /content && python -m pytest alphatrain/tests/test_model.py alphatrain/tests/test_observation.py -v --tb=short 2>&1 | tail -10

In [ ]:
# Verify both datasets
import torch

# Expert
ex = torch.load('/content/alphatrain/data/alphatrain_pairwise.pt', weights_only=True, map_location='cpu')
print(f"Expert: {ex['boards'].shape[0]:,} states, has_pairs={'good_boards' in ex}")
print(f"  max_score={ex['max_score']}, num_value_bins={ex['num_value_bins']}")
del ex

# Self-play
sp = torch.load('/content/alphatrain/data/selfplay_iter2.pt', weights_only=False, map_location='cpu')
print(f"\nSelf-play: {sp['observations'].shape[0]:,} states, {sp['n_games']} games")
print(f"  Obs: {sp['observations'].shape}, Policy: {sp['policy_targets'].shape}")
print(f"  Value range: {sp['value_targets'].min():.1f} - {sp['value_targets'].max():.1f}")
print(f"  Policy T: {sp.get('policy_temperature', 'N/A')}")
ent = -(sp['policy_targets'] * (sp['policy_targets'] + 1e-10).log()).sum(dim=-1).mean()
print(f"  Policy entropy: {ent:.2f} (T=0.3 should be ~1.2-1.5)")
del sp

In [ ]:
# Pillar 2g: Hybrid Interleaved Training
#
# Expert (1.31M states): pol_CE + 0.001 * (ranking + anchor)
# Self-play (~340K states, cycled): pol_CE + 0.001 * MSE
# Loss = (loss_expert + loss_selfplay) / 2
#
# LR=5e-5 (conservative refinement), 15 epochs, warm start from 2f
%cd /content
!python -m alphatrain.train_hybrid \
    --expert-file alphatrain/data/alphatrain_pairwise.pt \
    --selfplay-file alphatrain/data/selfplay_iter2.pt \
    --gpu-data --amp --compile \
    --value-bins 1 --val-weight 0.001 --rank-weight 1.0 --anchor-weight 0.001 \
    --resume alphatrain/data/pillar2f_best.pt --warm-start \
    --epochs 15 --batch-size 4096 --lr 5e-5 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2g_best.pt \
    --save-dir /content/checkpoints/pillar2g

In [ ]:
# Evaluate policy score (baseline from Pillar 2f was 315)
# Should stay >= 310 (no regression)
%cd /content
!python -m alphatrain.evaluate --player policy \
    --model /content/checkpoints/pillar2g/best.pt \
    --games 50 --seed 42

In [ ]:
# Copy final model to Drive
import shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2g/{f}'
    dst = f'{DRIVE}/pillar2g_{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')